# TIỀN XỬ LÝ HÀNG LOẠT TOÀN BỘ DATASET FG-NET (OFFLINE BATCH PREPROCESSING)

Notebook này thực hiện tiền xử lý offline cho **toàn bộ dataset FG-NET (~1002 ảnh)** mà **không cần GPU Diffusion** (tiết kiệm tài nguyên và thời gian chạy pipeline chính):
1. **Adaptive Padding (`cv2.BORDER_REPLICATE`)**: Nếu khuôn mặt chiếm >85% khung hình, mở rộng biên 20% bằng replicate để tránh co kéo khi face alignment.
2. **Kiểm tra Grayscale trước White Balance**:
   - Dùng hàm `is_effectively_grayscale(threshold=6.0)`.
   - Nếu là ảnh đen-trắng / xám: **Bỏ qua White Balance** để bảo toàn tông xám gốc, tránh bị ám màu sai.
   - Nếu là ảnh màu: Áp dụng **Shades of Gray (Minkowski p-norm, $p=6$, $\Delta_{\max} \le 35.0$)** để khử vàng/sepia mà không ám xanh trán.
3. **CodeFormer (`fidelity_weight=0.7`, `--face_upsample`)**: Phục hồi chi tiết khuôn mặt với fidelity weight cố định $w=0.7$.
4. **Bảo toàn tên file gốc & Fallback per-image**: Giữ nguyên tên file `{person_id}A{age}.JPG`, bọc `try / except` từng ảnh (nếu lỗi sẽ copy ảnh gốc), xuất log CSV và đóng gói file nén `FGNET_preprocessed_full.zip`.

## 1. Khai báo Đường dẫn Dữ liệu & Cấu hình

In [ ]:
import os
import sys
import shutil
import zipfile
import datetime

# ===== ĐƯỜNG DẪN DỮ LIỆU FG-NET GỐC (KAGGLE) =====
FGNET_DIR_CANDIDATES = [
    "/kaggle/input/datasets/menonkk/nckh-2025-2026/FGNET (1)/FGNET/images",
    "/kaggle/input/datasets/menonkk/nckh-2025-2026/FGNET/images",
    "/kaggle/input/datasets/menonkk/nckh-2025-2026",
    "d:/Data/project/nckh/data/FGNET (1)/FGNET/images",
    "d:/Data/project/nckh/data/FGNET/images"
]

def find_fgnet_dir():
    for cand in FGNET_DIR_CANDIDATES:
        if os.path.isdir(cand):
            # Kiểm tra xem có chứa file .jpg không
            files = [f for f in os.listdir(cand) if f.lower().endswith(('.jpg', '.jpeg'))]
            if len(files) >= 50:
                return cand
            # Hoặc tìm đệ quy
            for root, _, rfiles in os.walk(cand):
                jpgs = [f for f in rfiles if f.lower().endswith(('.jpg', '.jpeg'))]
                if len(jpgs) >= 50:
                    return root
    return None

FGNET_DIR = find_fgnet_dir()
assert FGNET_DIR is not None, "❌ Không tìm thấy thư mục ảnh FG-NET. Vui lòng kiểm tra lại cấu trúc input trên Kaggle!"

OUTPUT_DIR = "/kaggle/working/FGNET_preprocessed"
ZIP_OUTPUT_PATH = "/kaggle/working/FGNET_preprocessed_full.zip"
LOG_CSV_PATH = "/kaggle/working/preprocessing_log.csv"
TEMP_CODEFORMER_DIR = "/kaggle/working/temp_cf_batch"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(TEMP_CODEFORMER_DIR, exist_ok=True)

all_images = sorted([f for f in os.listdir(FGNET_DIR) if f.lower().endswith(('.jpg', '.jpeg'))])
print(f"✅ Thư mục ảnh FG-NET gốc: {FGNET_DIR}")
print(f"✅ Tổng số ảnh tìm thấy: {len(all_images)} ảnh")
print(f"✅ Thư mục xuất kết quả: {OUTPUT_DIR}")
print(f"✅ File nén đầu ra: {ZIP_OUTPUT_PATH}")
print(f"✅ File log CSV: {LOG_CSV_PATH}")

## 2. Cài đặt Thư viện & Chuẩn bị Pre-trained Weights CodeFormer

In [ ]:
# ===== 1. Cài đặt các gói phụ thuộc =====
!pip install -q insightface onnxruntime-gpu
!pip install -q basicsr facexlib gfpgan

# ===== 2. Chuẩn bị repository & weights CodeFormer =====
import subprocess
import glob

CODEFORMER_DIR = "/kaggle/working/CodeFormer"
if not os.path.exists(CODEFORMER_DIR):
    print("Cloning CodeFormer repository...")
    subprocess.run(["git", "clone", "https://github.com/sczhou/CodeFormer.git", CODEFORMER_DIR], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{CODEFORMER_DIR}/requirements.txt"], check=True)


def patch_functional_tensor(root_dir):
    """Quét đệ quy 1 thư mục basicsr, vá mọi file .py còn import functional_tensor lỗi."""
    patched = []
    for py_file in glob.glob(os.path.join(root_dir, "**", "*.py"), recursive=True):
        with open(py_file, "r", encoding="utf-8", errors="ignore") as f:
            content = f.read()
        if "torchvision.transforms.functional_tensor" in content:
            content = content.replace("torchvision.transforms.functional_tensor", "torchvision.transforms.functional")
            with open(py_file, "w", encoding="utf-8") as f:
                f.write(content)
            patched.append(py_file)
    return patched


# ===== 3a. Vá bản basicsr CÀI QUA PIP (dùng cho các bước khác của pipeline) =====
import importlib.util

spec = importlib.util.find_spec("basicsr")
if spec is not None and spec.submodule_search_locations:
    pip_basicsr_path = list(spec.submodule_search_locations)[0]
    patched = patch_functional_tensor(pip_basicsr_path)
    print(f"✅ [pip basicsr] Đã vá {len(patched)} file tại {pip_basicsr_path}")
else:
    print("⚠️ Không tìm thấy bản basicsr cài qua pip — bỏ qua bước này.")

# ===== 3b. Vá bản basicsr LOCAL đóng gói sẵn trong CodeFormer (bản THẬT SỰ được dùng
# khi chạy inference_codeformer.py, vì Python ưu tiên import bản cùng thư mục repo) =====
local_basicsr_dir = os.path.join(CODEFORMER_DIR, "basicsr")
if os.path.isdir(local_basicsr_dir):
    patched_local = patch_functional_tensor(local_basicsr_dir)
    print(f"✅ [CodeFormer/basicsr local] Đã vá {len(patched_local)} file tại {local_basicsr_dir}")

    # File version.py bị thiếu vì chưa từng chạy `python basicsr/setup.py develop`
    # (bước cài đặt chính thức của CodeFormer, tự sinh file này). Tạo trực tiếp thay vì
    # chạy toàn bộ setup.py (tránh rủi ro build phức tạp không cần thiết cho use-case này).
    version_file = os.path.join(local_basicsr_dir, "version.py")
    if not os.path.exists(version_file):
        with open(version_file, "w", encoding="utf-8") as f:
            f.write('__version__ = "1.4.2"\n__gitsha__ = "unknown"\n')
        print(f"✅ Đã tạo file version.py còn thiếu tại {version_file}")
    else:
        print("✅ version.py đã tồn tại, không cần tạo lại.")
else:
    print(f"⚠️ Không thấy thư mục {local_basicsr_dir} — CodeFormer có thể không vendor basicsr local ở bản này.")

# ===== 4. Tải weights facelib và codeformer =====
print("Đang tải pretrained weights CodeFormer & Facelib...")
subprocess.run([sys.executable, f"{CODEFORMER_DIR}/scripts/download_pretrained_models.py", "facelib"], check=True)
subprocess.run([sys.executable, f"{CODEFORMER_DIR}/scripts/download_pretrained_models.py", "CodeFormer"], check=True)

print("✅ Môi trường CodeFormer đã sẵn sàng!")

## 3. Định nghĩa Module Tiền xử lý (Adaptive Padding, Grayscale Check, Shades of Gray WB, CodeFormer)

In [ ]:
import cv2
import numpy as np
from PIL import Image
from typing import Tuple, List, Dict, Optional
import insightface
from insightface.app import FaceAnalysis

# Khởi tạo FaceAnalysis cho bước Adaptive Padding
print("Khởi tạo InsightFace FaceAnalysis...")
app_insight = FaceAnalysis(name="buffalo_l", allowed_modules=["detection"])
app_insight.prepare(ctx_id=0 if os.environ.get("CUDA_VISIBLE_DEVICES") else -1, det_size=(256, 256))

def apply_adaptive_padding(
    image_rgb: np.ndarray, 
    face_occupancy_thresh: float = 0.85, 
    pad_ratio: float = 0.20,
    border_mode: str = "replicate",
    embedder = None
) -> Tuple[np.ndarray, bool]:
    """
    Bước 1: Adaptive Padding nếu khuôn mặt chiếm > 85% chiều dài/rộng ảnh gốc.
    Dùng cv2.BORDER_REPLICATE để triệt tiêu hoàn toàn hoa văn hình thoi.
    Trả về: (ảnh_sau_xử_lý, was_padded)
    """
    h, w, _ = image_rgb.shape
    img_bgr = cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR)

    faces = embedder.get(img_bgr) if embedder is not None else []
    if len(faces) == 0:
        return image_rgb, False

    bbox = faces[0].bbox.astype(int)
    face_w = bbox[2] - bbox[0]
    face_h = bbox[3] - bbox[1]

    occ_w = face_w / float(w)
    occ_h = face_h / float(h)

    if occ_w > face_occupancy_thresh or occ_h > face_occupancy_thresh:
        pad_y = int(h * pad_ratio)
        pad_x = int(w * pad_ratio)
        cv2_mode = cv2.BORDER_REPLICATE if border_mode == "replicate" else cv2.BORDER_REFLECT
        padded = cv2.copyMakeBorder(
            image_rgb, pad_y, pad_y, pad_x, pad_x,
            cv2_mode
        )
        return padded, True

    return image_rgb, False


def is_effectively_grayscale(image_rgb: np.ndarray, threshold: float = 6.0) -> bool:
    """
    Bước 2a: Kiểm tra ảnh có phải là ảnh đen-trắng/xám thực tế không.
    Tính độ chênh lệch màu trung bình giữa các kênh R-G và G-B.
    Nếu chênh lệch < threshold (mặc định 6.0) -> Ảnh xám -> Bỏ qua White Balance.
    """
    img_float = image_rgb.astype(np.float64)
    diff_rg = np.abs(img_float[:, :, 0] - img_float[:, :, 1]).mean()
    diff_gb = np.abs(img_float[:, :, 1] - img_float[:, :, 2]).mean()
    return float((diff_rg + diff_gb) / 2.0) < threshold


def apply_white_balance(
    image_rgb: np.ndarray, 
    p: float = 6.0,
    max_shift_thresh: float = 35.0, 
    clamp_range: Tuple[float, float] = (0.75, 1.30)
) -> np.ndarray:
    """
    Bước 2b: Shades of Gray White Balance (Minkowski p-norm, p=6, Finlayson & Trezzi 2004).
    Áp dụng cho ảnh màu có hiện tượng ngả vàng/sepia của ảnh cũ.
    Giữ nguyên độ ấm tự nhiên của da, triệt tiêu vệt xanh trán nhờ Minkowski p=6 + dynamic alpha-blend.
    """
    img_float = image_rgb.astype(np.float32)

    norm_r = np.power(np.mean(np.power(img_float[:, :, 0], p)), 1.0 / p) + 1e-6
    norm_g = np.power(np.mean(np.power(img_float[:, :, 1], p)), 1.0 / p) + 1e-6
    norm_b = np.power(np.mean(np.power(img_float[:, :, 2], p)), 1.0 / p) + 1e-6

    mean_norm = (norm_r + norm_g + norm_b) / 3.0
    raw_gain_r = mean_norm / norm_r
    raw_gain_g = mean_norm / norm_g
    raw_gain_b = mean_norm / norm_b

    clamped_gain_r = float(np.clip(raw_gain_r, clamp_range[0], clamp_range[1]))
    clamped_gain_g = float(np.clip(raw_gain_g, clamp_range[0], clamp_range[1]))
    clamped_gain_b = float(np.clip(raw_gain_b, clamp_range[0], clamp_range[1]))

    wb_temp = np.empty_like(img_float)
    wb_temp[:, :, 0] = np.clip(img_float[:, :, 0] * clamped_gain_r, 0, 255)
    wb_temp[:, :, 1] = np.clip(img_float[:, :, 1] * clamped_gain_g, 0, 255)
    wb_temp[:, :, 2] = np.clip(img_float[:, :, 2] * clamped_gain_b, 0, 255)

    avg_r = float(np.mean(img_float[:, :, 0]))
    avg_g = float(np.mean(img_float[:, :, 1]))
    avg_b = float(np.mean(img_float[:, :, 2]))

    shift_r = abs(float(np.mean(wb_temp[:, :, 0])) - avg_r)
    shift_g = abs(float(np.mean(wb_temp[:, :, 1])) - avg_g)
    shift_b = abs(float(np.mean(wb_temp[:, :, 2])) - avg_b)
    max_shift = max(shift_r, shift_g, shift_b)

    if max_shift > max_shift_thresh:
        alpha = max_shift_thresh / (max_shift + 1e-6)
    else:
        alpha = 1.0

    blended = img_float * (1.0 - alpha) + wb_temp * alpha
    return np.clip(blended, 0, 255).astype(np.uint8)


def run_codeformer(
    image_rgb: np.ndarray,
    fidelity_weight: float = 0.7,
    unique_tag: str = "temp",
    temp_dir: str = "/kaggle/working/temp_cf_batch"
) -> np.ndarray:
    """
    Bước 3: Chạy CodeFormer inference trên 1 ảnh RGB với fidelity_weight = 0.7 cố định.
    Gọi CLI inference_codeformer.py --face_upsample.
    """
    codeformer_script = os.path.join(CODEFORMER_DIR, "inference_codeformer.py")
    if not os.path.exists(codeformer_script):
        raise FileNotFoundError(f"Không tìm thấy script CodeFormer tại: {codeformer_script}")

    in_dir = os.path.join(temp_dir, f"in_{unique_tag}")
    out_dir = os.path.join(temp_dir, f"out_{unique_tag}")
    os.makedirs(in_dir, exist_ok=True)
    os.makedirs(out_dir, exist_ok=True)

    in_file = os.path.join(in_dir, "input.png")
    cv2.imwrite(in_file, cv2.cvtColor(image_rgb, cv2.COLOR_RGB2BGR))

    cmd = [
        sys.executable, codeformer_script,
        "-w", str(fidelity_weight),
        "--input_path", in_file,
        "-o", out_dir,
        "--face_upsample"
    ]

    proc = subprocess.run(cmd, capture_output=True, text=True, timeout=90)
    if proc.returncode != 0:
        raise RuntimeError(f"CodeFormer CLI lỗi (code {proc.returncode}):\n{proc.stderr}")

    res_path = os.path.join(out_dir, "final_results", "input.png")
    if not os.path.exists(res_path):
        fin_dir = os.path.join(out_dir, "final_results")
        if os.path.exists(fin_dir) and os.listdir(fin_dir):
            res_path = os.path.join(fin_dir, os.listdir(fin_dir)[0])
        else:
            raise FileNotFoundError(f"Không tìm thấy kết quả CodeFormer tại: {out_dir}")

    res_bgr = cv2.imread(res_path)
    if res_bgr is None:
        raise ValueError(f"Không đọc được ảnh kết quả CodeFormer: {res_path}")

    # Dọn dẹp thư mục tạm
    try:
        shutil.rmtree(in_dir, ignore_errors=True)
        shutil.rmtree(out_dir, ignore_errors=True)
    except Exception:
        pass

    return cv2.cvtColor(res_bgr, cv2.COLOR_BGR2RGB)

## 4. Sanity Check: Kiểm tra hàm `is_effectively_grayscale()` trên các ảnh mẫu

In [ ]:
import os
import cv2
import numpy as np

# Kiểm tra trước trên 036A05.JPG, 060A05.JPG (đen-trắng), 015A09.JPG (có màu)
test_sanity_files = ["036A05.JPG", "060A05.JPG", "015A09.JPG", "036A18.JPG"]

print("=" * 80)
print(f"{'Tên File':<15} | {'Metric chênh lệch RGB':<22} | {'Ngưỡng':<8} | {'Kết luận':<15} | {'Xử lý WB'}")
print("-" * 80)

for fname in test_sanity_files:
    fpath = os.path.join(FGNET_DIR, fname)
    if not os.path.exists(fpath):
        print(f"{fname:<15} | {'Không tìm thấy file':<22} | {'--':<8} | {'--':<15} | --")
        continue

    img_bgr = cv2.imread(fpath)
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    img_float = img_rgb.astype(np.float64)
    metric = float((np.abs(img_float[:, :, 0] - img_float[:, :, 1]).mean() + 
                    np.abs(img_float[:, :, 1] - img_float[:, :, 2]).mean()) / 2.0)
    is_gray = is_effectively_grayscale(img_rgb, threshold=6.0)

    res_label = "Ảnh XÁM (True)" if is_gray else "Ảnh MÀU (False)"
    wb_action = "BỎ QUA White Balance" if is_gray else "Chạy Shades of Gray WB"
    print(f"{fname:<15} | {metric:<22.4f} | {'6.0':<8} | {res_label:<15} | {wb_action}")

print("=" * 80)
print("✅ Sanity check hoàn tất: Phân biệt chính xác ảnh xám/đen-trắng để không bị méo màu khi WB.")

## 4b. [KIỂM TRA TRƯỚC KHI CHẠY FULL] Xác minh trực quan 3 ảnh từng có bug

Chạy đúng pipeline thật (padding + grayscale-check + white balance + CodeFormer) trên 3 ảnh
từng phát hiện bug trước đây, hiển thị ảnh gốc / sau xử lý cạnh nhau để xác nhận bằng mắt
TRƯỚC KHI chạy batch full 1002 ảnh (tốn nhiều giờ CodeFormer).

- `047A05`: kỳ vọng KHÔNG còn hoa văn hình thoi (diamond pattern) quanh viền mặt
- `003A35`, `004A37`: kỳ vọng màu da tự nhiên, KHÔNG ngả xanh lục/cyan, và đặc biệt phải
  xác nhận `is_effectively_grayscale()` KHÔNG chấm nhầm 2 ảnh này là ảnh xám (nếu nhầm,
  White Balance sẽ bị bỏ qua hoàn toàn và màu sepia gốc sẽ không được sửa)

In [ ]:
import matplotlib.pyplot as plt

KNOWN_BUG_FILES = ["047A05.JPG", "003A35.JPG", "004A37.JPG"]

fig, axes = plt.subplots(len(KNOWN_BUG_FILES), 2, figsize=(10, 5 * len(KNOWN_BUG_FILES)))

for row, fname in enumerate(KNOWN_BUG_FILES):
    src_path = os.path.join(FGNET_DIR, fname)
    if not os.path.exists(src_path):
        candidates = [f for f in all_images if fname.split('.')[0].upper() in f.upper()]
        if candidates:
            fname = candidates[0]
            src_path = os.path.join(FGNET_DIR, fname)
        else:
            print(f"⚠️ KHÔNG TÌM THẤY: {fname} — kiểm tra lại tên file trong dataset")
            continue

    orig_bgr = cv2.imread(src_path)
    orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)

    padded_rgb, was_padded = apply_adaptive_padding(
        orig_rgb, face_occupancy_thresh=0.85, pad_ratio=0.20,
        border_mode="replicate", embedder=app_insight
    )
    is_gray = is_effectively_grayscale(padded_rgb, threshold=6.0)

    if is_gray:
        wb_rgb = padded_rgb
        wb_note = "⚠️ Bị CHẤM LÀ ẢNH XÁM → Bỏ QUA WHITE BALANCE"
    else:
        wb_rgb = apply_white_balance(padded_rgb, p=6.0, max_shift_thresh=35.0)
        wb_note = "Đã chạy Shades of Gray WB (p=6)"

    try:
        final_rgb = run_codeformer(
            wb_rgb, fidelity_weight=0.7,
            unique_tag=f"sanity_{fname.split('.')[0]}",
            temp_dir=TEMP_CODEFORMER_DIR
        )
        cf_note = "CodeFormer OK"
    except Exception as e:
        final_rgb = wb_rgb
        cf_note = f"CodeFormer LỖI (dùng ảnh trước CF): {e}"

    print(f"[{fname}] padded={was_padded} | grayscale={is_gray} | {wb_note} | {cf_note}")

    axes[row, 0].imshow(orig_rgb)
    axes[row, 0].set_title(f"{fname} — GỐC")
    axes[row, 0].axis("off")

    axes[row, 1].imshow(final_rgb)
    axes[row, 1].set_title(f"{fname} — SAU Xử LÝ")
    axes[row, 1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/sanity_check_known_bugs.png", dpi=150)
plt.show()

print("\n" + "=" * 80)
print("KIỂM TRA BẰNG MẮT:")
print("  - 047A05: có còn hoa văn hình thoi quanh viền mặt không?")
print("  - 003A35, 004A37: màu da có tự nhiên không, hay vẫn ngả xanh lục/cyan?")
print("  - Nếu dòng log có 'Bị CHẤM LÀ ẢNH XÁM': kiểm tra kỹ màu ảnh SAU Xử LÝ có còn")
print("    sepia/ngả vàng như ảnh GỐC không — nếu còn, ngưỡng threshold=6.0 đang sai,")
print("    cần hạ ngưỡng xuống hoặc bỏ hẳn bước is_effectively_grayscale cho 2 ảnh này.")
print("CHẮC CHẮN CHẠY BATCH FULL 1002 ẢNH SAU KHI XÁC NHẮN CẢ 3 ẢNH TRÊN ĐÚNG NHư KỲ VỌNG.")
print("=" * 80)

## 5. Chạy Batch Tiền Xử Lý trên Toàn Bộ Dataset FG-NET

In [ ]:
import csv
import time
from datetime import timedelta

print("=" * 80)
print(f"BẮT ĐẦU CHẠY BATCH PREPROCESSING TRÊN TOÀN BỘ {len(all_images)} ẢNH FG-NET")
print("=" * 80)

n_total = len(all_images)
n_success = 0
n_fallback = 0
n_grayscale = 0
n_padded = 0

log_records = []
t_start = time.time()

for idx, fname in enumerate(all_images, start=1):
    t_img_start = time.time()
    src_path = os.path.join(FGNET_DIR, fname)
    dst_path = os.path.join(OUTPUT_DIR, fname) # BẢO TOÀN NGUYÊN VẸN TÊN FILE GỐC

    was_padded = False
    was_grayscale = False
    codeformer_success = False
    fallback_to_original = False
    status_msg = ""

    try:
        orig_bgr = cv2.imread(src_path)
        if orig_bgr is None:
            raise ValueError(f"Không thể đọc file ảnh: {src_path}")
        orig_rgb = cv2.cvtColor(orig_bgr, cv2.COLOR_BGR2RGB)

        # 1. Adaptive Padding (replicate viền khi mặt chiếm >85% khung hình)
        padded_rgb, was_padded = apply_adaptive_padding(
            orig_rgb, face_occupancy_thresh=0.85, pad_ratio=0.20, border_mode="replicate", embedder=app_insight
        )
        if was_padded:
            n_padded += 1

        # 2. Kiểm tra Grayscale
        was_grayscale = is_effectively_grayscale(padded_rgb, threshold=6.0)
        if was_grayscale:
            wb_rgb = padded_rgb
            n_grayscale += 1
        else:
            wb_rgb = apply_white_balance(padded_rgb, p=6.0, max_shift_thresh=35.0)

        # 3. CodeFormer (fidelity_weight=0.7 cố định)
        safe_tag = f"img_{idx}_{fname.split('.')[0]}"
        cf_rgb = run_codeformer(wb_rgb, fidelity_weight=0.7, unique_tag=safe_tag, temp_dir=TEMP_CODEFORMER_DIR)
        codeformer_success = True

        # Lưu ảnh đã tiền xử lý thành công
        cv2.imwrite(dst_path, cv2.cvtColor(cf_rgb, cv2.COLOR_RGB2BGR))
        n_success += 1
        status_msg = f"OK [Pad:{was_padded}, Gray:{was_grayscale}]"

    except Exception as e:
        # XỬ LÝ LỖI PER-IMAGE: Copy ảnh gốc sang output, ghi log fallback, không dừng batch
        fallback_to_original = True
        codeformer_success = False
        n_fallback += 1
        shutil.copy2(src_path, dst_path)
        status_msg = f"FALLBACK ({e})"

    # Lưu log
    record = {
        "filename": fname,
        "was_padded": was_padded,
        "was_grayscale": was_grayscale,
        "codeformer_success": codeformer_success,
        "fallback_to_original": fallback_to_original
    }
    log_records.append(record)

    img_time = time.time() - t_img_start
    elapsed = time.time() - t_start
    avg_time = elapsed / idx
    eta = avg_time * (n_total - idx)

    if idx % 25 == 0 or idx == n_total or fallback_to_original:
        print(f"[{idx:>4}/{n_total}] {fname:<12} | {img_time:>4.1f}s | {status_msg:<35} | ETA: {timedelta(seconds=int(eta))}")

# Ghi toàn bộ ra file CSV log
with open(LOG_CSV_PATH, "w", newline="", encoding="utf-8") as f_csv:
    writer = csv.DictWriter(f_csv, fieldnames=["filename", "was_padded", "was_grayscale", "codeformer_success", "fallback_to_original"])
    writer.writeheader()
    writer.writerows(log_records)

print("=" * 80)
print(f"HOÀN TẤT TIỀN XỬ LÝ {n_total} ẢNH:")
print(f"  - Thành công: {n_success} ({n_success/n_total*100:.1f}%)")
print(f"  - Fallback về gốc: {n_fallback} ({n_fallback/n_total*100:.1f}%)")
print(f"  - Số ảnh được Adaptive Padding: {n_padded}")
print(f"  - Số ảnh Grayscale (bỏ qua WB): {n_grayscale}")
print(f"  - Tổng thời gian: {timedelta(seconds=int(time.time() - t_start))}")
print(f"  - File log CSV: {LOG_CSV_PATH}")
print("=" * 80)

## 6. Đóng gói ZIP & Thống kê Định lượng

In [ ]:
import pandas as pd

# 1. Đóng gói thư mục thành file ZIP
print(f"Đang nén thư mục {OUTPUT_DIR} thành {ZIP_OUTPUT_PATH}...")
shutil.make_archive(ZIP_OUTPUT_PATH.replace(".zip", ""), "zip", OUTPUT_DIR)

zip_size_mb = os.path.getsize(ZIP_OUTPUT_PATH) / (1024 * 1024)
print(f"✅ Đã tạo file ZIP thành công: {ZIP_OUTPUT_PATH} ({zip_size_mb:.2f} MB)")

# 2. Phân tích kết quả từ file log CSV
df_log = pd.read_csv(LOG_CSV_PATH)
print("\n" + "=" * 60)
print("BẢNG TỔNG KẾT PREPROCESSING LOG:")
print("=" * 60)
print(f"Tổng số file xử lý        : {len(df_log)}")
print(f"Số file hoàn thành đầy đủ : {(df_log['codeformer_success'] == True).sum()}")
print(f"Số file phải fallback gốc : {(df_log['fallback_to_original'] == True).sum()}")
print(f"Số ảnh được padding       : {(df_log['was_padded'] == True).sum()}")
print(f"Số ảnh grayscale          : {(df_log['was_grayscale'] == True).sum()}")
print(f"Tỷ lệ thành công          : {((df_log['codeformer_success'] == True).mean() * 100):.2f}%")
print("=" * 60)

# Hiển thị 10 dòng đầu của log
df_log.head(10)